In [1]:
import os

from hydra import compose, initialize, initialize_config_dir
from omegaconf import OmegaConf

from FantAIno.constants import ROOT_DIR

from FantAIno.data.melondy_classic import MelondyClassicDataset

from FantAIno.preprocessing.album_data.preprocessing import AlbumData_Preprocessor
from FantAIno.preprocessing.lyrics.preprocessing import Lyrics_Embeddings_PCA

from FantAIno.models.neural_network import NeuralNetworkModel

c:\Users\Owner\Documents\SillyProjects\FantAIno\FantAIno_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = MelondyClassicDataset(
    split="train",
    data_type="pytorch",
    album_data_preprocessing_pipeline=AlbumData_Preprocessor(),
    lyrics_preprocessing_pipeline=Lyrics_Embeddings_PCA(),
)

C:\Users\Owner\Documents\SillyProjects\FantAIno\FantAIno\preprocessing\album_data\preprocessing.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  album_data_df.fillna(0, inplace=True)
C:\Users\Owner\Documents\SillyProjects\FantAIno\FantAIno\data\melondy_classic.py:34: UserWarning: WARNING: Lyrics embeddings table has 3 missing embeddings out of 3120 entries.
  self.lyrics_embeddings_df = lyrics_preprocessing_pipeline(self.lyrics_embeddings_df)


Your PCA model has explained 85.07% of the total variance in the lyrics embeddings.
NOTE: The number of rows with missing data: 300 out of 3419.


In [3]:
model = NeuralNetworkModel()

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
c:\Users\Owner\Documents\SillyProjects\FantAIno\FantAIno_venv\Lib\site-packages\lightning\pytorch\trainer\connectors\logger_connector\logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [4]:
# context initialization
with initialize_config_dir(version_base=None, config_dir=os.path.join(ROOT_DIR, "hydra_run_config")):
    cfg = compose(config_name="NNRegressionModel")
    print(OmegaConf.to_yaml(cfg))

mode: train
dataset:
  _target_: FantAIno.data.melondy_classic.MelondyClassicDataset
  split: ${mode}
  data_type: numpy
  drop_cols: null
  openai_embedding_size: small
preprocessing:
  lyrics_preprocessing_pipeline:
    _target_: FantAIno.preprocessing.preprocessing.PreprocessingPipeline
    steps:
    - _target_: FantAIno.preprocessing.lyrics.preprocessing.Lyrics_Embeddings_PCA
      n_components: 100
  album_data_preprocessing_pipeline:
    _target_: FantAIno.preprocessing.preprocessing.PreprocessingPipeline
    steps:
    - _target_: FantAIno.preprocessing.album_data.preprocessing.AlbumData_Preprocessor
model:
  _target_: FantAIno.models.neural_network.NeuralNetworkModel
  mode: regression
  hidden_dims:
  - 128
  - 128
  - 128
  output_dim: 1
  activation:
    _target_: torch.nn.ReLU
  final_activation: null
  dropout: 0.2
  loss_fn:
    _target_: torch.nn.MSELoss
  optimizer: ${optimizer}
  trainer_dict: ${training.trainer}
  model_run_name: base
optimizer:
  _target_: torch.opt

In [3]:
X_train, y_train = df.FantAIno_df_X, df.FantAIno_df_y

In [ ]:
model.train_model(X_train, y_train)

c:\Users\Owner\Documents\SillyProjects\FantAIno\FantAIno_venv\Lib\site-packages\lightning\pytorch\loops\utilities.py:73: `max_epochs` was not set. Setting it to 1000 epochs. To train without an epoch limit, set `max_epochs=-1`.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
c:\Users\Owner\Documents\SillyProjects\FantAIno\FantAIno_venv\Lib\site-packages\lightning\pytorch\trainer\configuration_validator.py:70: You defined a `validation_step` but have no `val_dataloader`. Skipping val loop.


In [6]:
# before training
import torch
print(torch.isfinite(X_train).all())
print(torch.isfinite(y_train).all())
print(X_train.abs().max())
print(y_train.abs().max())

tensor(True)
tensor(True)
tensor(16220.9854)
tensor(10.)
